In [0]:
pip install -U langchain-google-genai python-dotenv tabulate langchain-tavily --quiet

In [0]:
%restart_python

In [0]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pyspark.sql import SparkSession
from langchain_tavily import TavilySearch
from pyspark.sql import functions as F




In [0]:
load_dotenv()

In [0]:
Spark = (
    SparkSession.builder.getOrCreate()
)

In [0]:
df_gold = (
    spark
    .table('db_analytics.precio_semanal_brent')
    .orderBy('semana_inicio', ascending=False)
    .filter(
    F.col("semana_inicio") < F.trunc(F.current_date(), "week")
)
    .limit(2)
    .toPandas()
)

display(df_gold)

In [0]:
tabla_markdown = df_gold.to_markdown(index=False) 
tabla_markdown

In [0]:
with open('prompts/agente_analisis.txt') as archivo:
    prompt_template = archivo.read()

prompt_template

In [0]:
tavily_en = TavilySearch(
    topic="news",
    search_depth="advanced",
    include_answer=False,
    include_raw_content=True,
    max_results=3,
    include_domains=[
        "reuters.com",
        "bloomberg.com",
        "spglobal.com",
        "argusmedia.com",
        "oilprice.com",
    ],
)

tavily_es = TavilySearch(
    topic="news",
    search_depth="advanced",
    include_answer=False,
    include_raw_content=True,
    max_results=3,
    include_domains=[
        "tn.com.ar",
        "infobae.com",
        "eleconomista.com.ar",  
    ],
)

In [0]:

fecha_inicio = df_gold["semana_inicio"].max().strftime("%Y-%m-%d")
fecha_fin = (df_gold["semana_fin"].max()+ pd.Timedelta(days=1)).strftime("%Y-%m-%d")

queries_en = [
    "Brent crude oil price",
    "oil OPEC supply production",
    "oil inventories EIA United States",
    "oil Iran Middle East geopolitical attacks",
]

queries_es = [
    "precio petróleo Brent",
    "petróleo OPEP oferta producción",
    "inventarios petróleo Estados Unidos",
    "petróleo Irán Medio Oriente ataques",
]

queries_arg = [
    "Argentina Vaca Muerta producción petróleo",
    "Vaca Muerta exportaciones petróleo oleoductos infraestructura",
    "Vaca Muerta petróleo RIGI "
]

def buscar_noticias(tavily, queries, fecha_inicio, fecha_fin):
    resultados = []

    for query in queries:

        busqueda = tavily.invoke({
            "query": query,
            "start_date": fecha_inicio,
            "end_date": fecha_fin,
        })

        if isinstance(busqueda, dict):
            resultados.extend(busqueda.get("results", []))
    return resultados

resultados_en = buscar_noticias(
    tavily_en,
    queries_en,
    fecha_inicio,
    fecha_fin
)

resultados_es = buscar_noticias(
    tavily_es,
    queries_es,
    fecha_inicio,
    fecha_fin
)

resultados_arg = buscar_noticias(
    tavily_es,
    queries_arg,
    fecha_inicio,
    fecha_fin
)

resultados = resultados_en + resultados_es + resultados_arg


# Eliminar duplicados

resultados_unicos = {}

for resultado in resultados:
    url = resultado.get("url")

    if url:
        resultados_unicos[url] = resultado

resultados = list(resultados_unicos.values())

noticias = "\n\n".join(
    f"""
Título: {resultado.get("title") or "Sin título"}

Contenido: {
    resultado.get("raw_content")
    or resultado.get("content")
    or "Sin contenido"
}

Fuente: {resultado.get("url") or "Sin URL"}
""".strip()

    for resultado in resultados
)

In [0]:
print("Cantidad total:", len(resultados))

for r in resultados:
    print(r.get("title"))
    print(r.get("url"))
    print()

In [0]:
# Incluimos la INFORMACION de la tabla c/las 2 semanas y las NEWS de Tavily


prompt = (
    prompt_template
    .replace("{datos_semanales}", tabla_markdown)
    .replace("{noticias}", noticias)
)

In [0]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    max_retries=0
)

respuesta = llm.invoke(prompt)

print(respuesta.text)

In [0]:
dbutils.jobs.taskValues.set(key="analisis", value=respuesta.text)